In [ ]:
# ==========================================
# 0. UNIVERSAL DEPENDENCY INSTALLATION
# ==========================================
import subprocess
import sys

def install_dependencies():
    packages = [
        "torch", "transformers", "accelerate", "sentencepiece", "protobuf",
        "scikit-learn", "scipy", "pandas", "numpy", "tqdm"
    ]
    for pkg in packages:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

install_dependencies()

# ==========================================
# 1. SETUP & DATA LOADING
# ==========================================
import pandas as pd
import numpy as np
import torch
import re
from transformers import pipeline
import gc
from tqdm.auto import tqdm

print("📂 Loading Hindi Level 1 Core Political data...")
# Adjust path to your actual Hindi Level 1 CSV
df = pd.read_csv('/kaggle/input/notebooks/shivanguniyal/removing-author-form-hindi/FINAL_Hindi_Pristine_Corpus.csv')

# Identify the text column (assuming boilerplate cleaning is finished and saved in 'final_clean_text' or 'clean_text')
text_col = 'final_clean_text' if 'final_clean_text' in df.columns else ('clean_text' if 'clean_text' in df.columns else 'article_text')
inference_col = text_col

# Assign a unique Article ID for traceability if not present
if 'article_id' not in df.columns:
    df['article_id'] = df.index

# 🚨 CRUCIAL FIX: Filter out rows that became empty or too short after cleaning
# This prevents the "ValueError: You must include at least one label and at least one sequence" crash.
initial_count = len(df)
df = df[df[inference_col].astype(str).str.strip().str.len() > 50].reset_index(drop=True)
df = df.dropna(subset=[inference_col])
print(f"✓ Filtered out {initial_count - len(df)} empty/short articles. {len(df):,} articles ready for NLI inference.")

device = 0 if torch.cuda.is_available() else -1
pipe = pipeline("zero-shot-classification", 
                model="MoritzLaurer/mDeBERTa-v3-base-mnli-xnli", 
                device=device, batch_size=16)

candidate_labels = ["critical", "neutral", "supportive"]
hypothesis_template = "The overall evaluative orientation conveyed by this news article is {}."
label_to_idx = {label: idx for idx, label in enumerate(candidate_labels)}

# ==========================================
# STAGE 1: TRUNCATED NLI (PRIMARY CLASSIFIER)
# ==========================================
print("\n--- STAGE 1: Fast Truncated Inference ---")
texts_trunc = [str(t)[:1500] for t in df[inference_col].tolist()]

# Initialize NumPy array for fast probability storage
trunc_probs = np.zeros((len(df), 3))

super_chunk_size = 500 # Adjusted for multilingual model memory limits
for i in tqdm(range(0, len(texts_trunc), super_chunk_size), desc="Stage 1 Progress"):
    batch = texts_trunc[i:i+super_chunk_size]
    
    # Safety check: ensure batch is not empty
    if not batch:
        continue
        
    res = pipe(batch, candidate_labels, hypothesis_template=hypothesis_template, multi_label=False, truncation=True)
    
    for j, r in enumerate(res):
        idx = i + j
        for label, score in zip(r['labels'], r['scores']):
            trunc_probs[idx, label_to_idx[label]] = score

# Vectorized assignment to DataFrame (Instantaneous)
df['trunc_critical'] = trunc_probs[:, label_to_idx['critical']]
df['trunc_neutral'] = trunc_probs[:, label_to_idx['neutral']]
df['trunc_supportive'] = trunc_probs[:, label_to_idx['supportive']]
df['trunc_label'] = [candidate_labels[idx] for idx in np.argmax(trunc_probs, axis=1)]
df['trunc_confidence'] = np.max(trunc_probs, axis=1)

# Calculate Margin (Winner - Runner Up) for Rule C
sorted_probs = np.sort(trunc_probs, axis=1)
df['trunc_margin'] = sorted_probs[:, -1] - sorted_probs[:, -2]

# ==========================================
# STAGE 2: EMPIRICAL ESCALATION RULE (RULE C)
# ==========================================
print("\n--- STAGE 2: Identifying Uncertain Articles (Rule C) ---")
# HINDI THRESHOLDS: Conf < 0.45 OR Margin < 0.15 
# (Empirically justified for mDeBERTa cross-lingual softmax dilution)
CONFIDENCE_THRESHOLD = 0.45
MARGIN_THRESHOLD = 0.15

escalation_mask = (df['trunc_confidence'] < CONFIDENCE_THRESHOLD) | (df['trunc_margin'] < MARGIN_THRESHOLD)

# Add explicit boolean flag for pipeline diagnostics
df['escalated'] = escalation_mask 

num_to_rerun = escalation_mask.sum()
print(f"✓ {num_to_rerun:,} articles ({num_to_rerun/len(df)*100:.1f}%) flagged for Hierarchical Escalation via Rule C.")

# ==========================================
# STAGE 3: HIERARCHICAL CLASSIFICATION (ADJUDICATION)
# ==========================================
def get_hierarchical_chunks(text, chunk_size=1000):
    text = str(text)
    # Hindi-specific sentence splitting: includes Danda (।) and Double Danda (॥)
    sentences = re.split(r'(?<=[.!?।॥])\s+', text)
    chunks, current_chunk = [], ""
    for sent in sentences:
        if len(current_chunk) + len(sent) > chunk_size:
            if current_chunk: chunks.append(current_chunk)
            current_chunk = sent
        else:
            current_chunk += " " + sent if current_chunk else sent
    if current_chunk: chunks.append(current_chunk)
    return chunks if chunks else [text]

# Initialize Hierarchical Columns with NaN
df['hier_critical'] = np.nan
df['hier_neutral'] = np.nan
df['hier_supportive'] = np.nan
df['hier_label'] = np.nan

if num_to_rerun > 0:
    print(f"\n--- STAGE 3: Hierarchical Adjudication on {num_to_rerun:,} articles ---")
    df_ambig = df[escalation_mask].copy()
    df_ambig['chunks'] = df_ambig[inference_col].apply(get_hierarchical_chunks)
    
    all_chunks = []
    chunk_mapping = [] 
    for i, (idx, row) in enumerate(df_ambig.iterrows()):
        for chunk in row['chunks']:
            all_chunks.append(chunk)
            chunk_mapping.append(idx) 

    print(f"  Processing {len(all_chunks):,} total chunks...")
    
    # Accumulators for Mean Pooling
    hier_sums = {idx: np.zeros(3) for idx in df_ambig.index}
    hier_counts = {idx: 0 for idx in df_ambig.index}

    for i in tqdm(range(0, len(all_chunks), super_chunk_size), desc="Stage 3 Progress"):
        batch = all_chunks[i:i+super_chunk_size]
        
        if not batch:
            continue
            
        res = pipe(batch, candidate_labels, hypothesis_template=hypothesis_template, multi_label=False, truncation=True, batch_size=8) 
        
        for j, r in enumerate(res):
            orig_idx = chunk_mapping[i + j]
            for label, score in zip(r['labels'], r['scores']):
                hier_sums[orig_idx][label_to_idx[label]] += score
            hier_counts[orig_idx] += 1

    # Safe write-back to main DataFrame using Pandas .loc to prevent index misalignment
    for orig_idx in df_ambig.index:
        if hier_counts[orig_idx] > 0:
            avg_probs = hier_sums[orig_idx] / hier_counts[orig_idx]
            df.loc[orig_idx, 'hier_critical'] = avg_probs[label_to_idx['critical']]
            df.loc[orig_idx, 'hier_neutral'] = avg_probs[label_to_idx['neutral']]
            df.loc[orig_idx, 'hier_supportive'] = avg_probs[label_to_idx['supportive']]
            df.loc[orig_idx, 'hier_label'] = candidate_labels[np.argmax(avg_probs)]

# ==========================================
# STAGE 4: CONSOLIDATION & EXPORT
# ==========================================
print("\n--- STAGE 4: Consolidating Final Labels ---")

# Final Label: Hierarchical if escalated, otherwise Truncated
df['final_label'] = df['trunc_label']
df.loc[escalation_mask, 'final_label'] = df.loc[escalation_mask, 'hier_label']

# Track the method used
df['inference_method'] = 'Truncated (Primary)'
df.loc[escalation_mask, 'inference_method'] = 'Hierarchical (Adjudicated)'

# Explicit disagreement calculation to avoid NaN comparison quirks
df['disagreement'] = False
df.loc[escalation_mask, 'disagreement'] = df.loc[escalation_mask, 'trunc_label'] != df.loc[escalation_mask, 'hier_label']

# ==========================================
# GENERATE STRATEGIC SPOT-CHECK
# ==========================================
print("\n--- Generating Blinded Spot Check ---")
# Over-sample the adjudicated/disagreement articles
spot_check_hier = df[df['escalated']].sample(min(50, num_to_rerun), random_state=42) if num_to_rerun > 0 else pd.DataFrame()
spot_check_trunc = df[~df['escalated']].groupby('final_label', group_keys=False).apply(
    lambda x: x.sample(min(len(x), 17), random_state=42)
)
spot_check = pd.concat([spot_check_hier, spot_check_trunc]).sample(frac=1, random_state=42)

# Use the cleaned text column for human coders
cols_to_keep_spot = ['article_id', 'date', 'newspaper', 'period', 'Topic', inference_col]
spot_check_blinded = spot_check[[c for c in cols_to_keep_spot if c in spot_check.columns]].copy()
spot_check_blinded['Human_Label'] = ""
spot_check_blinded['Coder_Notes'] = ""
spot_check_blinded.to_csv('/kaggle/working/Hindi_Spot_Check_Blinded_2024.csv', index=False)

# ==========================================
# SAVE MASTER DATASET
# ==========================================
# Drop heavy text/chunk columns, keep ALL metadata and probabilities
cols_to_drop = [inference_col, 'chunks'] if 'chunks' in df.columns else [inference_col]
df_master = df.drop(columns=cols_to_drop)

# Reorder columns to match the Reviewer's requested metadata table exactly
priority_cols = [
    'article_id', 'date', 'newspaper', 'period', 'Topic', 'Macro_Bucket',
    'trunc_label', 'trunc_critical', 'trunc_neutral', 'trunc_supportive', 'trunc_confidence', 'trunc_margin',
    'escalated', 'hier_label', 'hier_critical', 'hier_neutral', 'hier_supportive',
    'final_label', 'inference_method', 'disagreement'
]
# Add any remaining columns that weren't in the priority list
remaining_cols = [c for c in df_master.columns if c not in priority_cols]
df_master = df_master[priority_cols + remaining_cols]

df_master.to_csv('/kaggle/working/Hindi_Master_Processed_Level1_2024.csv', index=False)

# ==========================================
# PRINT FINAL SUMMARY
# ==========================================
print("\n" + "="*60)
print("📊 CASCADE INFERENCE SUMMARY")
print("="*60)
print(f"Total Articles: {len(df):,}")
print(f"Resolved via Truncated (High Confidence): {(~escalation_mask).sum():,}")
print(f"Escalated to Hierarchical (Uncertain): {num_to_rerun:,}")

if num_to_rerun > 0:
    disagreements = df['disagreement'].sum()
    print(f"Disagreements (Truncated vs Hierarchical): {disagreements:,} ({disagreements/num_to_rerun*100:.1f}% of escalated)")
print("="*60)

del pipe
gc.collect()
torch.cuda.empty_cache()
print("\n✓ Pipeline complete! Ready for the Morning-After Diagnostics.")